# Used Car Price Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# File paths
TRAIN_PATH = "PATH/TO/train.csv"        # <-- set this to your local copy of the training CSV
TARGET_COLUMN = "price"                # <-- target variable: selling price
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"  # organizers only — do not rely on this file existing locally

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [ ]:
df =pd.read_csv('/content/dataset-2 (1).csv')
df.head()
#load the dataset

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,"$27,890"
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,"$50,000"
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,"$47,645"
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,"$29,990"


## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and extreme price/mileage outliers.
- Document any rows/columns you drop and why.

In [34]:
def data_cleaning(df):
    df = df.copy()
    print("Missing values in each column:")
    print(df.isnull().sum())
    print()
    print("Duplicate rows:", df.duplicated().sum())

    if 'price' in df.columns and df['price'].dtype == 'object':
        df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

    if 'milage' in df.columns and df['milage'].dtype == 'object':
        df['milage'] = df['milage'].str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)

    return df

## 3. Feature Engineering

Transform the raw columns into features that are more useful for a regression model.

For example: calculating vehicle age from `model_year`, extracting horsepower/displacement from `engine`, or categorizing rare brands.

In [13]:
def feature_engineering(df):
    # Implement your feature engineering steps here
    df=df.copy()

    df['vehicle_age'] = 2026 - df['model_year']

    print("New feature created: 'vehicle_age'")
    print(df[['model_year', 'vehicle_age']].head())



    return df


    pass

## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables (e.g., One-Hot or Target Encoding).
- Scale/normalize continuous numerical features (e.g., `mileage`, `age`, engine specs).

In [36]:
def scale_features(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    num_cols = ['milage', 'vehicle_age']
    cols_to_scale = [col for col in num_cols if col in df.columns]

    for col in cols_to_scale:
        col_min = df[col].min()
        col_max = df[col].max()
        if col_max != col_min:
            df[col] = (df[col] - col_min) / (col_max - col_min)
        else:
            df[col] = 0.0

    print("Features successfully transformed and scaled")
    print(df[cols_to_scale].head())
    return df

In [18]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [19]:
def split_data(df):
    # Implement your data splitting logic here
    # This function should return three dataframes: train, validation, and test
    ...
    train_val, test_split = train_test_split(df, test_size=0.2, random_state=42)
    train_split, validation_split = train_test_split(train_val, test_size=0.25, random_state=42)

    print(f"Train set size     : {len(train_split)} rows")
    print(f"Validation set size: {len(validation_split)} rows")
    print(f"Test set size      : {len(test_split)} rows")

    return train_split, validation_split, test_split

## 6. Model Training, and Hyperparameter Tuning

- Train regression models on the train set (e.g., Ridge, Random Forest, XGBoost/LightGBM).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 7) to report **train** and **validation**
  metrics for each model: RMSE, MAE, R².
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_price(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, and returns price predictions from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that has not been touched yet.

In [48]:
from sklearn.ensemble import RandomForestRegressor

# 1. Clean and engineer features globally before splitting
df_cleaned = data_cleaning(df)
df_engineered = feature_engineering(df_cleaned)

train_split, validation_split, test_split = split_data(df_engineered)

# 2. Extract features (X) and target (y)
X_train = train_split.drop(columns=['price'])
y_train = train_split['price']

X_val = validation_split.drop(columns=['price'])
y_val = validation_split['price']

# 3. Apply the scaling and encoding logic directly to get consistent columns across splits
X_train_scaled = scale_features(X_train)

# 4. Define and fit our RandomForest Model directly on the scaled training features
model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model.fit(X_train_scaled, y_train)

# Keep track of the columns the model was trained on to align prediction inputs
trained_cols = X_train_scaled.columns.tolist()

def predict_price(df_input):
    cleaned = data_cleaning(df_input)
    engineered = feature_engineering(cleaned)
    if 'price' in engineered.columns:
        engineered = engineered.drop(columns=['price'])

    scaled = scale_features(engineered)

    # Align columns efficiently to avoid DataFrame fragmentation warning
    missing_cols = {col: 0 for col in trained_cols if col not in scaled.columns}
    if missing_cols:
        scaled = pd.concat([scaled, pd.DataFrame(missing_cols, index=scaled.index)], axis=1)

    scaled = scaled[trained_cols]
    return model.predict(scaled)

Missing values in each column:
brand             0
model             0
model_year        0
milage            0
fuel_type       126
engine            0
transmission      0
ext_col           0
int_col           0
accident         92
clean_title     479
price             0
dtype: int64

Duplicate rows: 0
New feature created: 'vehicle_age'
   model_year  vehicle_age
0        2015           11
1        2017            9
2        2008           18
3        2022            4
4        2023            3
Train set size     : 1923 rows
Validation set size: 642 rows
Test set size      : 642 rows
Features successfully transformed and scaled
        milage  vehicle_age
672   0.079003      0.18750
1245  0.294367      0.18750
2913  0.382754      0.96875
1541  0.387060      0.50000
2520  0.183996      0.21875


In [49]:
def predict_price(df):
    # Re-apply the exact same sequential preprocessing pipeline
    cleaned = data_cleaning(df)
    engineered = feature_engineering(cleaned)
    if 'price' in engineered.columns:
        engineered = engineered.drop(columns=['price'])

    scaled = scale_features(engineered)

    # Align columns efficiently to avoid DataFrame fragmentation warning
    missing_cols = {col: 0 for col in trained_cols if col not in scaled.columns}
    if missing_cols:
        scaled = pd.concat([scaled, pd.DataFrame(missing_cols, index=scaled.index)], axis=1)

    scaled = scaled[trained_cols]
    return model.predict(scaled)

## 7. Evaluate on the Held-Out Test Set

Run `predict_price()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [29]:
def evaluate_predictions(y_true, y_pred):
    # DO NOT modify
    # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [47]:
X_test = test_split.drop(columns=['price'])

y_test = test_split['price']
if y_test.dtype == 'object':
    y_test = y_test.str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

test_predictions = predict_price(X_test)

print("--- Held-Out Test Set Metrics ---")
evaluate_predictions(y_test, test_predictions)

Missing values in each column:
brand            0
model            0
model_year       0
milage           0
fuel_type       19
engine           0
transmission     0
ext_col          0
int_col          0
accident        17
clean_title     93
vehicle_age      0
dtype: int64

Duplicate rows: 0
New feature created: 'vehicle_age'
      model_year  vehicle_age
610         2019            7
3050        2023            3
3176        2001           25
2890        2020            6
2885        2015           11
Features successfully transformed and scaled
        milage  vehicle_age
610   0.145494     0.161290
3050  0.001883     0.032258
3176  0.334839     0.741935
2890  0.084544     0.129032
2885  0.371218     0.290323


/tmp/ipykernel_1405/227177372.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scaled[col] = 0
/tmp/ipykernel_1405/227177372.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scaled[col] = 0
/tmp/ipykernel_1405/227177372.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scaled[col] = 0
/tmp

--- Held-Out Test Set Metrics ---
--- Metrics ---
      RMSE: 33771.7176
       MAE: 14975.5008
        R2: 0.5581


{'RMSE': np.float64(33771.71755102317),
 'MAE': 14975.500778656069,
 'R2': 0.5581083781740751}

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])